# Session 20 Solutions: Combining Tables

Data Science with Python, Level 1, Stage 2: NumPy and pandas

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## Setup

In [1]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
customers = pd.read_csv(DATA_URL + "shop_customers.csv")
products = pd.read_csv(DATA_URL + "shop_products.csv")
orders = pd.read_csv(DATA_URL + "shop_orders.csv")

## In-Class Exercises

### In-Class Exercise 1: Product Names on Orders

In [2]:
lines = orders.merge(products, on="product_id")
lines.groupby("product")["quantity"].sum().sort_values(ascending=False).head(5)

product
Index Cards               89
Highlighters              86
Sticky Notes              84
Pen Set                   76
Statistics Made Simple    73
Name: quantity, dtype: int64

### In-Class Exercise 2: Spend per Segment

In [3]:
full = orders.merge(products, on="product_id").merge(customers, on="customer_id")
full["total"] = full["price"] * full["quantity"]
full.groupby("segment")["total"].agg(["sum", "mean"]).round(2)

,sum,mean
segment,,
Premium,8489.19,58.55
Regular,38791.94,66.65
Student,10152.76,60.43


### In-Class Exercise 3: Check the Join

In [4]:
check = orders.merge(customers, on="customer_id", how="left", indicator=True)
lost = check[check["_merge"] == "left_only"]
print(len(lost), lost["order_id"].tolist())

5 [70056, 70244, 70455, 70739, 70877]


### In-Class Exercise 4: Different Key Names

In [5]:
clients = customers.rename(columns={"customer_id": "client"})
m = orders.merge(clients, left_on="customer_id", right_on="client")
print(len(m), len(orders.merge(customers, on="customer_id")))

895 895


## Part A: Basic

### Exercise 1: Customer Names on Orders

In [6]:
m = orders.merge(customers, on="customer_id", how="left")
print(m[["order_id", "name", "city", "quantity"]].head())
print(m["city"].value_counts(dropna=False))

   order_id            name    city  quantity
0     70001  Khaled Darwish    Homs         4
1     70002       Leen Issa  Aleppo         1
2     70003      Firas Issa    Homs         3
3     70004       Lea Karam    Homs         4
4     70005     Tala Khoury  Aleppo         2
city
Damascus    304
Aleppo      269
Homs        139
Latakia      93
Tartus       90
NaN           5
Name: count, dtype: int64


### Exercise 2: Stack Two Tables

In [7]:
a = pd.DataFrame({"day": [1, 2, 3], "revenue": [120, 135, 150]})
b = pd.DataFrame({"day": [1, 2, 3], "revenue": [90, 80, 110]})
a["branch"], b["branch"] = "Campus", "Mall"
both = pd.concat([a, b], ignore_index=True)
print(both)
print(both.groupby("branch")["revenue"].sum())

   day  revenue  branch
0    1      120  Campus
1    2      135  Campus
2    3      150  Campus
3    1       90    Mall
4    2       80    Mall
5    3      110    Mall
branch
Campus    405
Mall      280
Name: revenue, dtype: int64


## Part B: Intermediate

### Exercise 3: Category Revenue by Month

In [8]:
lines = orders.merge(products, on="product_id")
lines["total"] = lines["price"] * lines["quantity"]
lines["month"] = pd.to_datetime(lines["order_date"]).dt.month
table = pd.pivot_table(lines, index="month", columns="category", values="total", aggfunc="sum").round(0)
print(table)
print("Biggest growth:", (table.loc[6] - table.loc[1]).idxmax())

category    Bags   Books  Electronics  Stationery
month                                            
1         2134.0  1503.0       9732.0       785.0
2         1733.0  1196.0       3586.0       546.0
3         1503.0  1376.0       3641.0       644.0
4         1693.0  1496.0       6530.0       731.0
5          969.0  1370.0       5273.0       401.0
6         1786.0  1476.0       6759.0       720.0
Biggest growth: Books


### Exercise 4: Best Customers

In [9]:
full = orders.merge(products, on="product_id").merge(customers, on="customer_id")
full["total"] = full["price"] * full["quantity"]
best = full.groupby(["customer_id", "name", "city", "segment"]).agg(
    orders=("order_id", "count"), spent=("total", "sum")).round(2)
best.sort_values("spent", ascending=False).head(10)

,,,,orders,spent
customer_id,name,city,segment,,
C276,Khaled Halabi,Aleppo,Regular,7,2310.92
C220,Nour Najjar,Damascus,Regular,8,1967.93
C317,Hala Zein,Aleppo,Regular,8,1758.36
C314,Ziad Issa,Tartus,Regular,7,1490.66
C329,Rana Karam,Aleppo,Regular,10,1223.48
C243,Fadi Nassar,Damascus,Premium,3,1172.86
C294,Nour Karam,Damascus,Regular,6,1096.34
C232,Sami Aziz,Tartus,Regular,11,1043.90
C334,Yara Zein,Tartus,Regular,12,1016.10


## Part C: Challenge

### Exercise 5: Long Weather Table

In [10]:
w = pd.read_csv(DATA_URL + "weather_2024.csv")[["date", "city", "temp_max", "temp_min"]]
long = w.melt(id_vars=["date", "city"], var_name="measure", value_name="value")
print(long.head())
print(pd.pivot_table(long, index="city", columns="measure", values="value", aggfunc="mean").round(1))
# A long table has one value per row, which suits groupby and charts that colour lines by a category.

         date       city   measure  value
0  2024-01-01  Port City  temp_max   18.0
1  2024-01-02  Port City  temp_max   20.5
2  2024-01-03  Port City  temp_max   13.2
3  2024-01-04  Port City  temp_max   20.8
4  2024-01-05  Port City  temp_max   17.0
measure        temp_max  temp_min
city                             
Mountain Town      17.0       8.4
Port City          23.9      15.5


## Part D: Cumulative Project: Cafe Sales Analysis

### Exercise 6: Stage 2 Review: The Cafe Menu Table

In [11]:
df = pd.read_csv(DATA_URL + "cafe_sales.csv")
menu = pd.DataFrame({
    "item": ["Espresso", "Latte", "Cappuccino", "Mocha", "Green Tea", "Black Tea", "Croissant",
             "Muffin", "Cookie", "Chicken Sandwich", "Veggie Sandwich", "Orange Juice"],
    "cost": [0.6, 0.9, 0.9, 1.1, 0.5, 0.4, 0.9, 1.0, 0.4, 2.6, 2.1, 1.4],
})
m = df.merge(menu, on="item", how="left", indicator=True)
assert (m["_merge"] == "both").all()
m["profit"] = (m["unit_price"] - m["cost"]) * m["quantity"]
print("Total profit:", round(m["profit"].sum(), 2))
print(m.groupby("branch")["profit"].sum().round(2))
print(m.groupby("item")["profit"].sum().sort_values(ascending=False).head(3).round(2))
# Coffee drinks bring most of the profit, especially lattes and cappuccinos.
# The Mall branch earns the least, so a coffee promotion there could pay off.

Total profit: 3974.4
branch
Campus      1429.8
Downtown    1639.6
Mall         905.0
Name: profit, dtype: float64
item
Latte               657.8
Chicken Sandwich    608.4
Cappuccino          520.0
Name: profit, dtype: float64


## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | The default join is inner: only matching keys are kept. |
| 2 | B | A left join keeps every row of the left table. |
| 3 | B | Merge joins on keys; concat puts tables one after another. |
| 4 | B | Each repeated key produces one row per match. |
| 5 | B | `melt` turns a wide table into a long one. |